# A minimal introduction to object-oriented programming

Object-oriented programming (OOP) groups data and behavior into objects. You do **not** need to turn every analysis into a class. In AAI, you will mainly encounter objects because libraries expose models, data frames, figures, and estimators through object-oriented interfaces.

## 1. Classes and objects

A **class** is a blueprint. An **object** (or instance) is one concrete value created from that class. Class names conventionally use `PascalCase`.

In [1]:
class AAIStudent:
  pass


student = AAIStudent()
print(student)
print(type(student))

<class '__main__.AAIStudent'>


## 2. `__init__`, `self`, attributes, and methods

`__init__` runs when an instance is created. `self` refers to that instance. Attributes store its state. Methods define behavior.

In [2]:
class AAIStudent:
  def __init__(self, name: str, program: str, score: float = 0.0):
    self.name = name
    self.program = program
    self.score = score

  def introduce(self) -> str:
    return f"I am {self.name} from {self.program}, currently surviving AAI."

  def has_passed(self) -> bool:
    return self.score >= 50


ada = AAIStudent("Ada", "Mathematics", 69)
print(ada.name)
print(ada.introduce())
print(ada.has_passed())

Ada
I am Ada from Mathematics, currently surviving AAI.
True


### Solution: add a method

Add a method named `score_message()` that returns a short f-string containing the student's name and score. The placeholder method keeps the cell executable.

**Explanation:** The instance method accesses `name` and `score` through `self` and combines them in an f-string.

In [3]:
class PracticeStudent:
  def __init__(self, name: str, score: float):
    self.name = name
    self.score = score

  ### --- Start Solution --- ###
  def score_message(self) -> str:
    return f"{self.name} scored {self.score} points."
  ### --- End Solution --- ###


practice_student = PracticeStudent("Grace", 67)
print(practice_student.score_message())

Grace scored 67 points.


## 3. Class attributes and class methods

An instance attribute belongs to one object. A class attribute is shared through the class. A class method receives `cls` and is often used as an alternative constructor.

In [ ]:
class AAIStudent:
  course_name = "Applied Artificial Intelligence"
  passing_score = 50

  def __init__(self, name: str, program: str, score: float = 0.0):
    self.name = name
    self.program = program
    self.score = score

  def has_passed(self) -> bool:
    return self.score >= self.passing_score

  @classmethod
  def from_record(cls, record: dict):
    return cls(record["name"], record["program"], record.get("score", 0.0))


record = {"name": "Luisa", "program": "Information Science", "score": 99, "owns_cats": True}
margaret = AAIStudent.from_record(record)
print(margaret.course_name)
print(margaret.has_passed())

Applied Artificial Intelligence
True


## 4. Properties and encapsulation

Python does not enforce strict private fields. A leading underscore communicates “internal use.” A property provides controlled attribute-like access. Here the setter prevents scores outside the meaningful range.

In [5]:
class AAIStudent:
  def __init__(self, name: str, score: float = 0.0):
    self.name = name
    self._score = 0.0
    self.score = score

  @property
  def score(self) -> float:
    return self._score

  @score.setter
  def score(self, value: float) -> None:
    if not 0 <= value <= 100:
      raise ValueError("score must be between 0 and 100")
    self._score = float(value)


guido = AAIStudent("Guido", 69)
guido.score = 70
print(guido.score)
# But you can still access the private attribute directly
print(guido._score)

70.0
70.0


If you use two underscores, python will "scramble" the internal classname.

In [6]:
class AAIStudent:
  def __init__(self, name: str, score: float = 0.0):
    self.name = name
    self.__score = 0.0
    self.score = score

  @property
  def score(self) -> float:
    return self.__score

  @score.setter
  def score(self, value: float) -> None:
    if not 0 <= value <= 100:
      raise ValueError("score must be between 0 and 100")
    self.__score = float(value)


guido = AAIStudent("Guido", 69)
guido.score = 70
print(guido.score)
# Now you cannot access the private attribute directly
print(guido.__score)

70.0


AttributeError: 'AAIStudent' object has no attribute '__score'

### Solution: property validation

Extend the setter so it also rejects non-numeric values. Keep the example input numeric so the unedited notebook remains runnable.

**Explanation:** The setter checks the type before performing a numeric comparison. Booleans are rejected explicitly because Python otherwise treats them as integers. Invalid types raise `TypeError`. Out-of-range numbers raise `ValueError`.

In [7]:
class ValidatedExperiment:
  def __init__(self, learning_rate: float):
    self.learning_rate = learning_rate

  @property
  def learning_rate(self) -> float:
    return self._learning_rate

  ### --- Start Solution --- ###
  @learning_rate.setter
  def learning_rate(self, value: float) -> None:
    if isinstance(value, bool) or not isinstance(value, (int, float)):
      raise TypeError("learning_rate must be numeric")
    if not 0 < value <= 1:
      raise ValueError("learning_rate must be in (0, 1]")
    self._learning_rate = float(value)
  ### --- End Solution --- ###


experiment = ValidatedExperiment(0.69)
print(experiment.learning_rate)

0.69


## 5. Magic methods

Names such as `__init__` and `__repr__` are sometimes called magic or dunder methods. Python calls them for built-in operations. `__repr__` should give a useful developer-facing representation.

In [8]:
class Experiment:
  def __init__(self, name: str, accuracy: float):
    self.name = name
    self.accuracy = accuracy

  def __repr__(self) -> str: # repr = representation, str = string
    return f"Experiment(name={self.name!r}, accuracy={self.accuracy:.3f})"

  def __lt__(self, other) -> bool: # lt = less than, gt = greater than, le = less than or equal to, ge = greater than or equal to, eq = equal to, ne = not equal to
    if not isinstance(other, Experiment):
      return NotImplemented
    return self.accuracy < other.accuracy


experiments = [Experiment("baseline", 0.67), Experiment("coffee-powered", 0.69)]
print(experiments)
print(sorted(experiments))

[Experiment(name='baseline', accuracy=0.670), Experiment(name='coffee-powered', accuracy=0.690)]
[Experiment(name='baseline', accuracy=0.670), Experiment(name='coffee-powered', accuracy=0.690)]


## 6. Inheritance and polymorphism

Inheritance creates a specialized class from a base class. Polymorphism means different objects can respond to the same method call. Prefer composition over deep inheritance trees. Nobody wants to debug a 14-level family tree.

In [9]:
class CourseParticipant:
  def __init__(self, name: str):
    self.name = name

  def role_message(self) -> str:
    return f"{self.name} participates in AAI."


class TeachingAssistant(CourseParticipant):
  def __init__(self, name: str, tutorial_group: str):
    super().__init__(name)
    self.tutorial_group = tutorial_group

  def role_message(self) -> str:
    return f"{self.name} rescues tutorial group {self.tutorial_group}."


participants = [CourseParticipant("Ada"), TeachingAssistant("Victor", "T2")]
for participant in participants:
  print(participant.role_message())

Ada participates in AAI.
Victor rescues tutorial group T2.


### Solution: inheritance

Create a `Lecturer` subclass that overrides `role_message()`. It may mention live demos, Bayreuth, or an entirely fictional confidence level of 420%.

**Explanation:** `Lecturer` inherits the constructor from `CourseParticipant`, so it can be instantiated with a name immediately. Overriding `role_message()` provides behavior specific to the subclass.

In [10]:
### --- Start Solution --- ###
class Lecturer(CourseParticipant):
  def role_message(self) -> str:
    return f"{self.name} starts a live demo in Bayreuth with 420% confidence."
### --- End Solution --- ###


lecturer = Lecturer("The Lecturer")
print(lecturer.role_message())

The Lecturer starts a live demo in Bayreuth with 420% confidence.


# Optional: dataclasses and inner classes

`@dataclass` generates common methods for classes that mainly store data. This is often clearer than writing a repetitive `__init__`.

In [11]:
from dataclasses import dataclass, field


@dataclass
class ModelRun:
  name: str
  accuracy: float
  tags: list[str] = field(default_factory=list)


run = ModelRun("baseline", 0.69, ["demo", "offline"])
print(run)

ModelRun(name='baseline', accuracy=0.69, tags=['demo', 'offline'])


An inner class is defined inside another class. It can express a tightly related helper type, but it is uncommon in day-to-day data analysis and often less readable than two normal classes.

In [12]:
class AAIPlatform:
  class Status:
    READY = "ready"
    RUNNING = "running"


print(AAIPlatform.Status.READY)

ready


## Takeaway

You should now be able to read the basic class-based APIs used by data-science libraries:

1. Construct an object: `model = SomeModel(...)`
2. Call a method: `model.fit(...)`
3. Read an attribute: `model.some_attribute`

Use a class when state and behavior naturally belong together. For a simple transformation, a function is often the better tool.

Next: **00_03_Pandas.ipynb**.